# Week 3, Day 4: pandas II: grouping, reshaping and the psychometric curve

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w003/d4.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def make_session(seed=0, n=600, bias=0.05, sigma=0.15, lapse=0.06):
    rng = np.random.default_rng(seed)
    contrasts = np.array([0, 0.0625, 0.125, 0.25, 1.0])
    side = rng.choice([-1, 1], n)
    c = rng.choice(contrasts, n)
    signed = side * c
    p_right = lapse + (1 - 2 * lapse) / (1 + np.exp(-(signed - bias) / sigma))
    choice = np.where(rng.random(n) < p_right, 1, -1)
    timeout = rng.random(n) < 0.04
    rt = 0.25 + 0.6 * np.exp(-3 * c) * rng.lognormal(0, 0.35, n)
    df = pd.DataFrame({
        "trial": np.arange(1, n + 1),
        "contrast_left": np.where(side == -1, c, 0.0),
        "contrast_right": np.where(side == 1, c, 0.0),
        "choice": np.where(timeout, 0, choice),
        "rt": np.where(timeout, np.nan, rt.round(3)),
        "feedback": np.where(timeout, -1, np.where(choice == side, 1, -1)),
    })
    df["signed_contrast"] = df["contrast_right"] - df["contrast_left"]
    df["chose_right"] = df["choice"] == 1
    return df

df = make_session()
valid = df[df["choice"] != 0]            # drop no-response trials
print(len(valid), "valid trials")

### Exercise: The psychometric table

Make `psych`: for each signed contrast, the number of valid trials `n`, the fraction choosing right `p_right`, and its standard error `sem` = $\sqrt{p(1-p)/n}$ (the standard error of a proportion). Then plot it with error bars.

In [ ]:
psych = valid.groupby("signed_contrast").agg(
    n=...,
    p_right=...,
)
psych["sem"] = ...

fig, ax = plt.subplots(figsize=(5, 3))
ax.errorbar(psych.index, psych["p_right"], yerr=psych["sem"], fmt="o", capsize=3)
ax.set_xlabel("signed contrast"); ax.set_ylabel("P(choose right)")
plt.show()
psych

In [ ]:
import numpy as np
def test_index():
    "psych has one row per signed contrast, in order"
    assert list(psych.index) == sorted(valid["signed_contrast"].unique())

def test_columns():
    "psych has columns n, p_right and sem"
    assert set(["n", "p_right", "sem"]) <= set(psych.columns)

def test_values():
    "p_right and its standard error are correct"
    g = valid.groupby("signed_contrast")["chose_right"]
    assert np.allclose(psych["p_right"], g.mean())
    p, n = g.mean(), g.size()
    assert np.allclose(psych["sem"], np.sqrt(p * (1 - p) / n))

test_index()
test_columns()
test_values()
print("All checks passed")

In [ ]:
sessions = pd.concat(
    [make_session(seed=s, bias=b).assign(session=s, mouse=m)
     for s, b, m in [(1, 0.10, "CSHL_001"), (2, 0.08, "CSHL_001"), (3, -0.12, "UCL_004"), (4, -0.09, "UCL_004")]],
    ignore_index=True,
)
mice = pd.DataFrame({"mouse": ["CSHL_001", "UCL_004"], "lab": ["cshl", "ucl"], "sex": ["F", "M"]})
sessions = sessions.merge(mice, on="mouse", how="left")    # adds lab and sex to every trial
sessions = sessions[sessions["choice"] != 0]
print(sessions.shape)
sessions.head()

### Exercise: One column per mouse

Use `pd.pivot_table` to make `wide`: rows are signed contrast, columns are mouse, values are the mean of `chose_right`. Then plot one line per mouse. Which mouse is biased towards which side?

In [ ]:
wide = pd.pivot_table(sessions, ...)

ax = wide.plot(marker="o", figsize=(5, 3))
ax.set_xlabel("signed contrast"); ax.set_ylabel("P(choose right)")
plt.show()
wide

In [ ]:
import numpy as np
def test_shape():
    "wide has one row per signed contrast and one column per mouse"
    assert list(wide.columns) == ["CSHL_001", "UCL_004"]
    assert len(wide) == sessions["signed_contrast"].nunique()

def test_values():
    "values are the fraction choosing right"
    m = sessions[sessions["mouse"] == "UCL_004"].groupby("signed_contrast")["chose_right"].mean()
    assert np.allclose(wide["UCL_004"], m)

test_shape()
test_values()
print("All checks passed")

### Exercise: Fit bias, threshold and lapse

Complete `psychometric(c, bias, sigma, lapse)` (one lapse rate for both sides) and fit it to `psych` with `curve_fit`. Use starting values `p0=[0, 0.2, 0.05]` and `bounds=([-1, 0.01, 0], [1, 1, 0.5])`. Store the fitted values in `params`. The simulation used bias 0.05, threshold 0.15 and lapse 0.06: how close do you get?

In [ ]:
from scipy.optimize import curve_fit

def psychometric(c, bias, sigma, lapse):
    return ...

x = psych.index.to_numpy()
y = psych["p_right"].to_numpy()
err = np.maximum(psych["sem"].to_numpy(), 0.02)    # avoid zero error bars
params, cov = curve_fit(psychometric, x, y, sigma=err, p0=[0, 0.2, 0.05], bounds=([-1, 0.01, 0], [1, 1, 0.5]))
print("bias, threshold, lapse =", np.round(params, 3))

cc = np.linspace(-1, 1, 200)
fig, ax = plt.subplots(figsize=(5, 3))
ax.errorbar(x, y, yerr=psych["sem"], fmt="o", capsize=3, label="data")
ax.plot(cc, psychometric(cc, *params), label="fit")
ax.set_xlabel("signed contrast"); ax.set_ylabel("P(choose right)"); ax.legend(frameon=False)
plt.show()

In [ ]:
import numpy as np
def test_params():
    "params holds bias, threshold and lapse"
    assert len(params) == 3

def test_recovered():
    "the fit recovers the true values used in the simulation"
    bias, sigma, lapse = params
    assert abs(bias - 0.05) < 0.05, f"bias {bias:.3f}, true 0.05"
    assert abs(sigma - 0.15) < 0.07, f"threshold {sigma:.3f}, true 0.15"
    assert abs(lapse - 0.06) < 0.05, f"lapse {lapse:.3f}, true 0.06"

test_params()
test_recovered()
print("All checks passed")